# AI Learning Series, Day 12: Build an Image Classifier

**Tech Boy Deepak, AI Learning Series, Deep Learning Part 4**

Today we teach a computer to **read handwritten digits (0 to 9)**: the same skill a machine needs to read marks written on an answer sheet, or the PIN code on a postal cover.

We will use a small **CNN (Convolutional Neural Network)** and the famous **MNIST** dataset: 70,000 small photos of handwritten digits.

**The plan, block by block**
1. Setup
2. Load the dataset
3. Look at one image *as numbers*, and at a few images as pictures
4. Prepare the data (divide by 255)
5. Build a small CNN
6. Compile (choose the loss and the optimizer)
7. Train (`fit`)
8. Test on 10,000 new images (`evaluate`)
9. Predict one image and see the percentages
10. See the images it got wrong
11. Try it yourself: a hand-made filter, and a peek inside the trained CNN
12. Homework: Fashion-MNIST
13. *(optional)* Dense vs CNN, and shifted digits: check the comparison from the video yourself

**How to run:** click **Runtime → Run all**. A normal CPU runtime is fine (training takes a few minutes). For more speed: **Runtime → Change runtime type → T4 GPU**.

> Your exact numbers (accuracy, time) will be a little different from the video. That is normal: it depends on the computer and on tiny random differences.

## Block 1: Setup
We load the tools: **NumPy** (numbers), **Matplotlib** (pictures) and **Keras** (the deep learning library, running on TensorFlow). We also fix the random seed so the starting weights are the same every time.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers

keras.utils.set_random_seed(42)   # same random starting weights every run
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)

## Block 2: Load the dataset
MNIST comes built into Keras. It is already split into **60,000 training images** (to learn from) and **10,000 test images** (kept aside, to check the model on images it has never seen). Each image comes with its **label**: the correct digit.

In [ ]:
dataset = keras.datasets.mnist    # HOMEWORK (Block 12): change this one line to keras.datasets.fashion_mnist

(x_train, y_train), (x_test, y_test) = dataset.load_data()

if dataset is keras.datasets.fashion_mnist:
    class_names = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat",
                   "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
else:
    class_names = [str(d) for d in range(10)]

print("Training images:", x_train.shape)   # (60000, 28, 28) -> 60,000 images of 28 x 28 pixels
print("Test images:    ", x_test.shape)
print("First 10 labels:", y_train[:10])

## Block 3: An image is just a grid of numbers
A computer does not "see" a picture. It sees **28 × 28 = 784 numbers**. Each number is the brightness of one pixel: **0 = black, 255 = full white**. In MNIST the digit is written in white on a black background.

In [ ]:
idx = 0
print("Label:", class_names[y_train[idx]])
for row in x_train[idx]:
    print("".join(f"{v:4d}" for v in row))   # print every pixel value, row by row

In [ ]:
# The same numbers drawn as a picture, plus a few more examples
fig, axes = plt.subplots(3, 6, figsize=(9, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i], cmap="gray")
    ax.set_title(f"Label: {class_names[y_train[i]]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Block 4: Prepare the data
Two small jobs:
1. **Divide by 255** so every pixel is between 0 and 1. Small, similar-sized numbers make training smoother and faster.
2. **Add a "channel" dimension.** A CNN expects *height × width × channels*. Grayscale has **1** channel (a colour photo would have 3: R, G, B).

In [ ]:
x_train = (x_train / 255.0).astype("float32")[..., np.newaxis]
x_test  = (x_test  / 255.0).astype("float32")[..., np.newaxis]
print("New shape:", x_train.shape, "| min:", x_train.min(), "| max:", x_train.max())

## Block 5: Build a small CNN
Read it top to bottom, like a factory line:
- **Conv2D(32, 3×3)**: 32 small 3×3 filters slide over the image. Each one learns to spot one simple pattern (an edge, a curve) and makes a **feature map**. ReLU keeps the positives.
- **MaxPooling2D(2×2)**: shrinks each feature map by keeping the strongest value in every 2×2 box.
- A second **Conv2D(64) + MaxPooling2D**: combines simple patterns into bigger shapes (loops, corners, strokes).
- **Flatten**: lays all the feature maps out in one long line of numbers.
- **Dense(64)**: a normal fully connected layer, like Day 9 and Day 10.
- **Dense(10, softmax)**: 10 output neurons, one per class. Softmax turns them into **percentages that add up to 100%**.

Nobody types in the filter values. They start random and are **learned** during training.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation="relu"),   # 32 filters -> 32 feature maps (26 x 26)
    layers.MaxPooling2D((2, 2)),                    # 26 x 26 -> 13 x 13
    layers.Conv2D(64, (3, 3), activation="relu"),   # 64 filters -> 64 feature maps (11 x 11)
    layers.MaxPooling2D((2, 2)),                    # 11 x 11 -> 5 x 5
    layers.Flatten(),                               # 5 x 5 x 64 = 1,600 numbers in a line
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),         # 10 percentages, one per class
])
model.summary()

## Block 6: Compile
We tell the model *how* to learn (Day 11 vocabulary):
- **optimizer = "adam"**: a smarter version of gradient descent.
- **loss = "sparse_categorical_crossentropy"**: the same log loss from Day 10, just for 10 classes: **−ln(the percentage given to the correct digit)**. "Sparse" only means our labels are plain numbers like 7, not lists.
- **metrics = ["accuracy"]**: we *train* on loss and *report* accuracy.

In [ ]:
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

## Block 7: Train
`fit` runs the loop from Day 10: **forward pass → loss → backprop → update**, again and again.
- **epochs=5**: go through all the training images 5 times.
- **batch_size=128**: update the weights after every 128 images.
- **validation_split=0.1**: keep 10% of the training images aside to check progress after each epoch.

In [ ]:
start = time.time()
history = model.fit(x_train, y_train,
                    epochs=5,
                    batch_size=128,
                    validation_split=0.1)
print(f"Training time: {time.time() - start:.0f} seconds")

In [ ]:
# Accuracy after each epoch: is the model getting better?
plt.plot(range(1, 6), history.history["accuracy"], marker="o", label="training images")
plt.plot(range(1, 6), history.history["val_accuracy"], marker="o", label="validation images")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(); plt.title("Learning progress")
plt.show()

## Block 8: Test on 10,000 new images
The real exam. These test images were **never** used in training.

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc * 100:.2f}%   (test loss: {test_loss:.4f})")
print(f"That is about {round(test_acc * 10000)} correct out of 10,000.")

## Block 9: Predict one image
Pick any test image (change `idx`). The model gives 10 percentages, and the biggest one is its answer.

In [ ]:
idx = 0                                   # try 1, 2, 3 ... any number up to 9999
probs = model.predict(x_test[idx:idx + 1], verbose=0)[0]

for name, p in zip(class_names, probs):
    print(f"{name:>10}: {p * 100:7.3f}%")
print("Model says:", class_names[probs.argmax()], "| True answer:", class_names[y_test[idx]])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3))
ax1.imshow(x_test[idx, :, :, 0], cmap="gray"); ax1.axis("off")
ax1.set_title(f"True: {class_names[y_test[idx]]}")
ax2.bar(class_names, probs * 100); ax2.set_ylabel("%"); ax2.set_title("Softmax percentages")
ax2.tick_params(axis="x", labelrotation=45)
plt.tight_layout(); plt.show()

## Block 10: The images it got wrong
No model is perfect. Let's look at some mistakes. Many of them are digits that even a person would find confusing.

In [ ]:
all_probs = model.predict(x_test, verbose=0)
preds = all_probs.argmax(axis=1)
wrong = np.where(preds != y_test)[0]
print(f"Wrong: {len(wrong)} out of {len(y_test)} test images")

fig, axes = plt.subplots(2, 5, figsize=(10, 5.5))
for ax, i in zip(axes.flat, wrong[:10]):
    ax.imshow(x_test[i, :, :, 0], cmap="gray"); ax.axis("off")
    ax.set_title(f"True {class_names[y_test[i]]} | Said {class_names[preds[i]]}\n"
                 f"({all_probs[i].max() * 100:.0f}% sure)", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# The model's top 2 guesses for each mistake: often the right answer is its second choice
for i in wrong[:10]:
    first, second = all_probs[i].argsort()[::-1][:2]
    print(f"Image {i:4d}: true {class_names[y_test[i]]:>3} | model: "
          f"{class_names[first]} {all_probs[i][first] * 100:.0f}%, "
          f"{class_names[second]} {all_probs[i][second] * 100:.0f}%")

## Block 11 (optional): Try it yourself
### (a) A hand-made filter
Here **we** choose a 3×3 vertical-edge filter and slide it over one image, exactly like the drawing in the video. In the real CNN, the filters are **learned**, not hand-made. This is only to see what "a filter makes a feature map" looks like.

In [ ]:
def apply_filter(image, kernel):
    # Slide a 3x3 kernel over the image: multiply 9 pairs of numbers and add them up
    h, w = image.shape
    out = np.zeros((h - 2, w - 2))
    for i in range(h - 2):
        for j in range(w - 2):
            out[i, j] = np.sum(image[i:i + 3, j:j + 3] * kernel)
    return out

vertical   = np.array([[1, 0, -1],
                       [1, 0, -1],
                       [1, 0, -1]])
horizontal = vertical.T                   # the same filter turned sideways

img = x_test[0, :, :, 0]
maps = {"Original": img,
        "Vertical filter + ReLU": np.maximum(apply_filter(img, vertical), 0),
        "Horizontal filter + ReLU": np.maximum(apply_filter(img, horizontal), 0)}

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, (title, m) in zip(axes, maps.items()):
    ax.imshow(m, cmap="gray"); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()

### (b) A peek inside the trained CNN
These are 8 of the 32 feature maps that the **first Conv2D layer** made for the same image. Each one comes from a filter the model **learned by itself** during training.

In [ ]:
first_layer = keras.Model(inputs=model.inputs, outputs=model.layers[0].output)
fmaps = first_layer.predict(x_test[0:1], verbose=0)[0]     # shape (26, 26, 32)

fig, axes = plt.subplots(1, 8, figsize=(14, 2.2))
for k, ax in enumerate(axes):
    ax.imshow(fmaps[:, :, k], cmap="gray"); ax.set_title(f"filter {k}"); ax.axis("off")
plt.tight_layout(); plt.show()

## Block 12: Homework, Fashion-MNIST
Go to **Block 2** and change one line:

```python
dataset = keras.datasets.fashion_mnist
```

Then click **Runtime → Run all**. Same code, same CNN, but now it learns 10 kinds of clothing (T-shirt, trouser, sneaker, bag...). Is the accuracy higher or lower than for digits? Which classes does it confuse? Tell me in the comments!

## Block 13 (optional): Dense vs CNN, and shifted digits
In the video we compared our CNN with a plain **Dense** network. This block lets you check that comparison yourself.
1. We train a **Dense** network with no filters (Flatten → Dense(128) → Dense(10)), with the same settings as our CNN: 5 epochs, batch size 128, 10% validation.
2. We move **every test digit 3 pixels to the right** and test both models again. For us it is the same digit. Is it the same for the models?

Run Blocks 1 to 8 first: this block uses the trained CNN (`model`). It trains one more small network, so it takes a little extra time. Your numbers may be a little different from the video.

In [ ]:
# Block 13 (optional): train a plain Dense network the same way, then test both models on shifted digits
keras.utils.set_random_seed(42)                    # fixed seed, so you get the same Dense network every run
dense_model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Flatten(),                              # 28 x 28 grid -> one line of 784 numbers (no filters)
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax"),
])
dense_model.compile(optimizer="adam",
                    loss="sparse_categorical_crossentropy",
                    metrics=["accuracy"])
dense_model.fit(x_train, y_train, epochs=5, batch_size=128,
                validation_split=0.1, verbose=0)  # same 5 epochs and batch size as our CNN
print("Dense network parameters:", f"{dense_model.count_params():,}")

# Move every test image 3 pixels to the right (the empty strip on the left stays black)
shift = 3
x_test_shifted = np.zeros_like(x_test)
x_test_shifted[:, :, shift:, :] = x_test[:, :, :-shift, :]

print(f"\n{'Model':>6} | normal test digits | shifted {shift} pixels")
for name, m in [("Dense", dense_model), ("CNN", model)]:
    normal = m.evaluate(x_test, y_test, verbose=0)[1] * 100
    moved = m.evaluate(x_test_shifted, y_test, verbose=0)[1] * 100
    print(f"{name:>6} | {normal:17.2f}% | {moved:15.2f}%")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 3))
ax1.imshow(x_test[0, :, :, 0], cmap="gray"); ax1.set_title("Test image 0"); ax1.axis("off")
ax2.imshow(x_test_shifted[0, :, :, 0], cmap="gray"); ax2.set_title(f"Shifted {shift} px right"); ax2.axis("off")
plt.tight_layout(); plt.show()